<a href="https://colab.research.google.com/github/august9999/hello-python/blob/master/generates%20dots_circles%20DXF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
!{sys.executable} -m pip install Pillow ezdxf numpy

In [12]:
import numpy as np
from PIL import Image, ImageDraw
import ezdxf
import os

# --- Configuration Parameters ---
IMAGE_PATH = 'input_image.png'  # Path to your input image
OUTPUT_DXF_PATH = 'output_circles.dxf' # Output DXF file name

RECTANGLE_HEIGHT_MM = 2.0  # Height of the rectangular area in millimeters
RECTANGLE_WIDTH_MM = 8.0   # Width of the rectangular area in millimeters
CIRCLE_DIAMETER_MICROMETERS = 15.0 # Diameter of circles in micrometers
GRAYSCALE_THRESHOLD = 255 # Grayscale value (0-255). Below this is 'dark' enough to place a circle.
                          # Lower values mean darker areas are recognized.
MAX_PLACEMENT_ATTEMPTS = 50000 # Maximum attempts to place circles randomly
MIN_PROBABILITY_FOR_LIGHT_GREY = 0.01 # Minimum probability for the lightest gray areas (0.0 to 1.0)

# --- Derived Parameters ---
RECTANGLE_HEIGHT_UM = RECTANGLE_HEIGHT_MM * 1000 # Convert mm to micrometers
RECTANGLE_WIDTH_UM = RECTANGLE_WIDTH_MM * 1000 # Convert mm to micrometers
CIRCLE_RADIUS_UM = CIRCLE_DIAMETER_MICROMETERS / 2.0

# Determine image resolution for grayscale sampling
# User requested 1 pixel = 15 micrometers
PIXELS_PER_MICROMETER_SCALE = 15.0

IMAGE_WIDTH_PX = int(RECTANGLE_WIDTH_UM / PIXELS_PER_MICROMETER_SCALE)
IMAGE_HEIGHT_PX = int(RECTANGLE_HEIGHT_UM / PIXELS_PER_MICROMETER_SCALE)

if IMAGE_WIDTH_PX == 0:
    IMAGE_WIDTH_PX = 1
if IMAGE_HEIGHT_PX == 0:
    IMAGE_HEIGHT_PX = 1

print(f"Calculated Image Resolution for Grayscale Sampling: {IMAGE_WIDTH_PX}x{IMAGE_HEIGHT_PX} pixels (1 pixel = {PIXELS_PER_MICROMETER_SCALE} micrometers)")

# --- Load and process the image ---
try:
    img = Image.open(IMAGE_PATH).convert('L') # Open image and convert to grayscale ('L' mode)
    # Resize the image to match our derived pixel dimensions for sampling
    img = img.resize((IMAGE_WIDTH_PX, IMAGE_HEIGHT_PX), Image.Resampling.LANCZOS)
    img_array = np.array(img)
    print(f"Loaded and resized image to {img_array.shape[1]}x{img_array.shape[0]} pixels.")
except FileNotFoundError:
    print(f"Error: Image file not found at {IMAGE_PATH}. Please make sure the image is uploaded and the path is correct.")
    import sys
    sys.exit(1)
except Exception as e:
    print(f"An error occurred while loading or processing the image: {e}")
    import sys
    sys.exit(1)

# --- Initialize DXF document ---
doc = ezdxf.new('R2010')  # Create a new DXF R2010 document
msp = doc.modelspace() # Get the model space

# --- Place circles randomly ---
placed_circles_count = 0
placed_circle_centers = [] # Store centers of placed circles for overlap checking

# Scaling factors to convert micrometers to image pixels for grayscale sampling
micrometer_to_pixel_x = IMAGE_WIDTH_PX / RECTANGLE_WIDTH_UM
micrometer_to_pixel_y = IMAGE_HEIGHT_PX / RECTANGLE_HEIGHT_UM

print(f"Attempting to place circles (Max Attempts: {MAX_PLACEMENT_ATTEMPTS})...")

for attempt in range(MAX_PLACEMENT_ATTEMPTS):
    # Generate random coordinates for the circle center within the rectangular bounds
    # Ensure the entire circle fits within the rectangle
    random_x_um = np.random.uniform(CIRCLE_RADIUS_UM, RECTANGLE_WIDTH_UM - CIRCLE_RADIUS_UM)
    random_y_um = np.random.uniform(CIRCLE_RADIUS_UM, RECTANGLE_HEIGHT_UM - CIRCLE_RADIUS_UM)

    # Convert random micrometer coordinates to image pixel coordinates for grayscale sampling
    px = int(random_x_um * micrometer_to_pixel_x)
    py = int(random_y_um * micrometer_to_pixel_y)

    # Ensure pixel coordinates are within image bounds (due to int casting and floating point)
    px = max(0, min(px, IMAGE_WIDTH_PX - 1))
    py = max(0, min(py, IMAGE_HEIGHT_PX - 1))

    grayscale_value = img_array[py, px]

    # Calculate placement probability based on grayscale value
    # Darker areas (lower grayscale_value) should have higher probability.
    # We normalize the grayscale_value to be between 0 and GRAYSCALE_THRESHOLD.
    # If grayscale_value is 0, probability is 1. If grayscale_value is GRAYSCALE_THRESHOLD, probability is MIN_PROBABILITY_FOR_LIGHT_GREY.
    if grayscale_value < GRAYSCALE_THRESHOLD:
        normalized_grayscale = grayscale_value / GRAYSCALE_THRESHOLD
        placement_probability = 1.0 - (normalized_grayscale * (1.0 - MIN_PROBABILITY_FOR_LIGHT_GREY))
        # Ensure probability is not less than MIN_PROBABILITY_FOR_LIGHT_GREY
        placement_probability = max(placement_probability, MIN_PROBABILITY_FOR_LIGHT_GREY)
    else:
        placement_probability = 0.0 # No chance to place a circle if it's too light

    # Use the probability to decide if a circle should be considered for placement
    if np.random.rand() < placement_probability:
        is_overlapping = False
        # Check for overlaps with already placed circles
        for cx, cy in placed_circle_centers:
            distance = np.sqrt((random_x_um - cx)**2 + (random_y_um - cy)**2)
            if distance < CIRCLE_DIAMETER_MICROMETERS: # Circles touch or overlap if distance < diameter
                is_overlapping = True
                break

        if not is_overlapping:
            msp.add_circle((random_x_um, random_y_um), CIRCLE_RADIUS_UM, dxfattribs={'layer': 'CIRCLES'})
            placed_circle_centers.append((random_x_um, random_y_um))
            placed_circles_count += 1

print(f"Placed {placed_circles_count} circles in the DXF file.")

# --- Save DXF document ---
try:
    doc.saveas(OUTPUT_DXF_PATH)
    print(f"DXF file saved successfully to {OUTPUT_DXF_PATH}")
except Exception as e:
    print(f"An error occurred while saving the DXF file: {e}")

Calculated Image Resolution for Grayscale Sampling: 533x133 pixels (1 pixel = 15.0 micrometers)
Loaded and resized image to 533x133 pixels.
Attempting to place circles (Max Attempts: 50000)...
Placed 14511 circles in the DXF file.
DXF file saved successfully to output_circles.dxf


In [5]:
!pip install ezdxf